In [31]:
#import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
#from sklearn.metrics import classification_report, confusion_matrix, roc_curve, roc_auc_score
#from sklearn.model_selection import cross_val_score
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

from sklearn.linear_model import LogisticRegression
from sklearn import svm
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.dummy import DummyClassifier
from xgboost import XGBClassifier

import mlflow
import pickle

In [2]:
data = pd.read_csv("../data/processed/transfer_modeling_dataset.csv")
print(data['transfer_success'].value_counts(normalize=True)*100)

transfer_success
0    82.801498
1    17.198502
Name: proportion, dtype: float64


In [3]:
NON_FEATURE_COLUMNS = {
    "transfer_key",
    "player_id",
    "player_full_name",
    "player_name",
    "source_club_id",
    "destination_club_id",
    "source_club_name",
    "destination_club_name",
    "source_club_dataset_name",
    "destination_club_dataset_name",
    "follow_up_window_end",
    "pre_transfer_market_value_date",
    "market_value_180d_prior_date",
    "market_value_365d_prior_date",
    "target_end_market_value_date",
    "target_failure_reason",
    "target_is_eligible",
    "exclusion_reason",
    "date_of_birth",
    "source_country_name",
    "destination_country_name",
    "pre_transfer_market_value_club_id",
    "pre_transfer_market_value_league_id",
    "valuation_cutoff_180d",
    "valuation_cutoff_365d",
    "player_window_start_180d",
    "player_window_end_180d",
    "player_window_start_365d",
    "player_window_end_365d",
    "source_api_cache_file",
    "destination_api_cache_file",
    "player_current_market_value",
    "highest_market_value_in_eur",
    "transfer_season",
    "source_competition_name",
    "destination_competition_name",
    "source_competition_name",
    "destination_competition_name",
    "target_destination_matches_24m",
    "target_destination_minutes_24m",
    "target_destination_goals_24m",
    "target_destination_assists_24m",
    "target_end_market_value",
    "target_market_value_delta_24m"
}
print(len(NON_FEATURE_COLUMNS))
data.drop(columns=NON_FEATURE_COLUMNS, inplace=True)

42


In [4]:
# 1. Select the row (e.g., the first row at index 0)
row_data = data.iloc[0]

# 2. Combine values and data types into a single table
result = pd.DataFrame({
    'Value': row_data,
    'DataType': data.dtypes
})

print(result)


                                      Value DataType
transfer_date                    2018-07-01   object
source_competition_id                   IT1   object
destination_competition_id              IT1   object
age_at_transfer                        35.0  float64
position                         Goalkeeper   object
...                                     ...      ...
destination_api_context_matched           0    int64
is_same_league_move                       1    int64
is_same_country_move                      1    int64
season_start_year                      2018    int64
transfer_success                          0    int64

[61 rows x 2 columns]


In [5]:
# Returns a list of column names
cols_with_nan = data.columns[data.isna().any()].tolist()
print(cols_with_nan)
print(len(cols_with_nan))

['source_competition_id', 'age_at_transfer', 'foot', 'transfer_fee', 'market_value_in_eur', 'market_value_180d_prior', 'market_value_365d_prior', 'market_value_change_180d', 'market_value_change_365d', 'transfer_fee_to_market_value_ratio', 'transfer_fee_minus_market_value', 'player_goals_per90_180d_pre', 'player_assists_per90_180d_pre', 'player_goals_per90_365d_pre', 'player_assists_per90_365d_pre', 'source_squad_size', 'source_average_age', 'source_foreigners_percentage', 'source_national_team_players', 'source_stadium_seats', 'destination_average_age', 'destination_foreigners_percentage', 'source_win_rate_365d_pre', 'source_points_per_match_365d_pre', 'source_goal_diff_per_match_365d_pre', 'destination_win_rate_365d_pre', 'destination_points_per_match_365d_pre', 'destination_goal_diff_per_match_365d_pre', 'source_api_cached_matches', 'source_api_cached_win_rate', 'source_api_cached_goal_diff_per_match', 'destination_api_cached_matches', 'destination_api_cached_win_rate', 'destination

In [6]:
rows_with_nan = data[data.isna().any(axis=1)]
print(len(rows_with_nan))

6675


In [7]:
# 1. Define your threshold
threshold = 0.2*len(data)  # For example, 20% of the total number of rows

# 2. Identify columns with NaNs > threshold
# data.isna().sum() returns a Series where index is column name and value is NaN count
nan_counts = data.isna().sum()
cols_to_log = nan_counts[nan_counts > threshold]

# 3. Print or Log the results
if not cols_to_log.empty:
    print(f"Columns with more than {threshold} NaNs:")
    print(cols_to_log)
else:
    print("No columns exceed the NaN threshold.")

Columns with more than 1335.0 NaNs:
source_competition_id                         1501
player_goals_per90_180d_pre                   3048
player_assists_per90_180d_pre                 3048
player_goals_per90_365d_pre                   2485
player_assists_per90_365d_pre                 2485
source_squad_size                             1501
source_average_age                            1532
source_foreigners_percentage                  1538
source_national_team_players                  1501
source_stadium_seats                          1501
source_win_rate_365d_pre                      1527
source_points_per_match_365d_pre              1527
source_goal_diff_per_match_365d_pre           1527
source_api_cached_matches                     6675
source_api_cached_win_rate                    6675
source_api_cached_goal_diff_per_match         6675
destination_api_cached_matches                6674
destination_api_cached_win_rate               6674
destination_api_cached_goal_diff_per_match    

In [8]:
data = data.dropna(axis=1,thresh=0.8*len(data))

In [9]:
# Returns a Series with column names and their respective NaN counts
nan_counts = data.isna().sum()

# Filter to show only columns that have more than 0 NaNs
print(nan_counts[nan_counts > 0])

age_at_transfer                               2
foot                                         12
transfer_fee                                559
market_value_in_eur                           5
market_value_180d_prior                     226
market_value_365d_prior                     539
market_value_change_180d                    226
market_value_change_365d                    539
transfer_fee_to_market_value_ratio          559
transfer_fee_minus_market_value             559
destination_average_age                      23
destination_foreigners_percentage            23
destination_win_rate_365d_pre               314
destination_points_per_match_365d_pre       314
destination_goal_diff_per_match_365d_pre    314
dtype: int64


In [10]:
metadata = {
    'features': data.columns.tolist(),
    'data_types': data.dtypes.tolist(),
}
metadata_df = pd.DataFrame(metadata)
metadata_df.to_csv("../data/metadata.csv", index=False)

In [11]:
print(len(data.columns))
rows_with_nan = data[data.isna().any(axis=1)]
print(len(rows_with_nan))
print(len(rows_with_nan)/len(data))

42
1135
0.1700374531835206


In [12]:
# 1. Identify columns with at least one NaN
cols_with_nans = data.columns[data.isna().any()]

# 2. Build a summary table
nan_summary = pd.DataFrame({
    'Column': cols_with_nans,
    'Data Type': data   [cols_with_nans].dtypes.values,
    'NaN Count': data[cols_with_nans].isna().sum().values,
    '% Missing': (data  [cols_with_nans].isna().mean().values * 100).round(2)
})

print("Summary of columns with missing values:")
print(nan_summary.to_string(index=False))

Summary of columns with missing values:
                                  Column Data Type  NaN Count  % Missing
                         age_at_transfer   float64          2       0.03
                                    foot    object         12       0.18
                            transfer_fee   float64        559       8.37
                     market_value_in_eur   float64          5       0.07
                 market_value_180d_prior   float64        226       3.39
                 market_value_365d_prior   float64        539       8.07
                market_value_change_180d   float64        226       3.39
                market_value_change_365d   float64        539       8.07
      transfer_fee_to_market_value_ratio   float64        559       8.37
         transfer_fee_minus_market_value   float64        559       8.37
                 destination_average_age   float64         23       0.34
       destination_foreigners_percentage   float64         23       0.34
           

In [ ]:
data['age_at_transfer'].fillna(data['age_at_transfer'].median(), inplace=True)
data['foot'].fillna(data['foot'].mode()[0], inplace=True)
data['transfer_fee'].fillna(data['transfer_fee'].median(), inplace=True)
data['market_value_in_eur'].fillna(data['market_value_in_eur'].median(), inplace=True)
data['market_value_180d_prior'].fillna(data['market_value_180d_prior'].median(), inplace=True)
data['market_value_365d_prior'].fillna(data['market_value_365d_prior'].median(), inplace=True)
data['market_value_change_180d'].fillna(data['market_value_change_180d'].median(), inplace=True)
data['market_value_change_365d'].fillna(data['market_value_change_365d'].median(), inplace=True)
data['transfer_fee_to_market_value_ratio'].fillna(data['transfer_fee_to_market_value_ratio'].median(), inplace=True)
data['transfer_fee_minus_market_value'].fillna(data['transfer_fee_minus_market_value'].median(), inplace=True)
data['destination_average_age'].fillna(data['destination_average_age'].median(), inplace=True)
data['destination_foreigners_percentage'].fillna(data['destination_foreigners_percentage'].median(), inplace=True)
data['destination_win_rate_365d_pre'].fillna(data['destination_win_rate_365d_pre'].median(), inplace=True)
data['destination_points_per_match_365d_pre'].fillna(data['destination_points_per_match_365d_pre'].median(), inplace=True)
data['destination_goal_diff_per_match_365d_pre'].fillna(data['destination_goal_diff_per_match_365d_pre'].median(), inplace=True)

In [14]:
# 1. Identify columns with at least one NaN
cols_with_nans = data.columns[data.isna().any()]

# 2. Build a summary table
nan_summary = pd.DataFrame({
    'Column': cols_with_nans,
    'Data Type': data   [cols_with_nans].dtypes.values,
    'NaN Count': data[cols_with_nans].isna().sum().values,
    '% Missing': (data  [cols_with_nans].isna().mean().values * 100).round(2)
})

print("Summary of columns with missing values:")
print(nan_summary.to_string(index=False))

Summary of columns with missing values:
Empty DataFrame
Columns: [Column, Data Type, NaN Count, % Missing]
Index: []


In [15]:
import sys 
from pathlib import Path
ROOT =Path.cwd().resolve().parent
if str(ROOT) not in sys.path:
    sys.path.append(str(ROOT))

from src.features.build_features import chronological_split
data_dict = chronological_split(data)
#train_data, test_data = train_test_split(data, test_size=0.15, shuffle=True, random_state=42)
print(data_dict['train']['transfer_success'].value_counts(normalize=True)*100)
print(data_dict['test']['transfer_success'].value_counts(normalize=True)*100)
data_dict['train'].to_csv("../data/model/train_data.csv", index=False)
data_dict['test'].to_csv("../data/model/test_data.csv", index=False) 

transfer_success
0    81.870719
1    18.129281
Name: proportion, dtype: float64
transfer_success
0    90.419162
1     9.580838
Name: proportion, dtype: float64


In [16]:
# I will use simple validation technique just divide train data into train and validation
#train_data, val_data = train_test_split(train_data, test_size=0.15, shuffle=True, random_state=42)
print(data_dict['train']['transfer_success'].value_counts(normalize=True)*100)
print(data_dict['val']['transfer_success'].value_counts(normalize=True)*100)

transfer_success
0    81.870719
1    18.129281
Name: proportion, dtype: float64
transfer_success
0    79.52048
1    20.47952
Name: proportion, dtype: float64


In [17]:
y_train =   data_dict['train']['transfer_success']
X_train = data_dict['train'].drop(columns=['transfer_success'])
y_val =   data_dict['val']['transfer_success']
X_val = data_dict['val'].drop(columns=['transfer_success'])

In [18]:
# 1. Get the count of each data type
type_counts = data_dict['train'].dtypes.value_counts()

print("--- Data Type Counts ---")
print(type_counts)

# 2. See exactly which columns belong to each type
print("\n--- Columns by Type ---")
for dtype in type_counts.index:
    cols = data_dict['train'].select_dtypes(include=[dtype]).columns.tolist()
    print(f"{dtype}: {cols}")

--- Data Type Counts ---
float64    26
int64       9
object      7
Name: count, dtype: int64

--- Columns by Type ---
float64: ['age_at_transfer', 'height_in_cm', 'transfer_fee', 'market_value_in_eur', 'pre_transfer_market_value', 'market_value_180d_prior', 'market_value_365d_prior', 'market_value_change_180d', 'market_value_change_365d', 'transfer_fee_to_market_value_ratio', 'transfer_fee_minus_market_value', 'player_matches_180d_pre', 'player_minutes_180d_pre', 'player_goals_180d_pre', 'player_assists_180d_pre', 'player_matches_365d_pre', 'player_minutes_365d_pre', 'player_goals_365d_pre', 'player_assists_365d_pre', 'destination_average_age', 'destination_foreigners_percentage', 'source_matches_365d_pre', 'destination_matches_365d_pre', 'destination_win_rate_365d_pre', 'destination_points_per_match_365d_pre', 'destination_goal_diff_per_match_365d_pre']
int64: ['destination_squad_size', 'destination_national_team_players', 'destination_stadium_seats', 'source_api_context_matched', 'de

In [19]:
# now I will handle categorical features with frequency encoding as sklearn does not support categorical features
# directly and I want to avoid one-hot encoding due to high cardinality of some features.
categorical_cols = X_train.select_dtypes(include=['object']).columns
for col in categorical_cols:
    freq_encoding = X_train[col].value_counts() / len(X_train)
    X_train[col] = X_train[col].map(freq_encoding)
    X_val[col] = X_val[col].map(freq_encoding).fillna(0)  # Handle unseen categories in validation set
    data_dict['test'][col] = data_dict['test'][col].map(freq_encoding).fillna(0)  # Handle unseen categories in test set

data_dict['test'].to_csv("../data/model/test_data_encoded.csv", index=False)


In [ ]:

experiment = mlflow.set_experiment("Training Models")
experiment_id = experiment.experiment_id
#mlflow.sklearn.autolog()
results_df = pd.DataFrame(columns=["model_name", "accuracy", "precision", "recall", "f1_score"])

# first I will try dummy classifier to set a baseline for the model performance
with mlflow.start_run(run_name="ZeroR Classifier", experiment_id=experiment_id):
    ZeroR = DummyClassifier(strategy='most_frequent')
    ZeroR.fit(X_train, y_train)
    y_pred = ZeroR.predict(X_val)
    print("ZeroR Classifier")
    mlflow.log_metric("accuracy", accuracy_score(y_val, y_pred))
    mlflow.log_metric("precision", precision_score(y_val, y_pred))
    mlflow.log_metric("recall", recall_score(y_val, y_pred))
    mlflow.log_metric("f1_score", f1_score(y_val, y_pred))
    mlflow.sklearn.log_model(sk_model=ZeroR, name="zero_r_classifier")
    with open("../models/zero_r_classifier.pkl", "wb") as f:
        pickle.dump(ZeroR, f)

# Append the results to the DataFrame
results_df = results_df.append({
    "model_name": "ZeroR Classifier",
    "accuracy": accuracy_score(y_val, y_pred),
    "precision": precision_score(y_val, y_pred),
    "recall": recall_score(y_val, y_pred),
    "f1_score": f1_score(y_val, y_pred)
}, ignore_index=True)

2026/05/03 14:49:46 INFO mlflow.utils.autologging_utils: Created MLflow autologging run with ID '4ad4f19dd07c44569c5ac11a01544a43', which will track hyperparameters, performance metrics, model artifacts, and lineage information for the current sklearn workflow
2026/05/03 14:49:46 WARNING mlflow.utils.autologging_utils: MLflow autologging encountered a warning: "c:\Users\Ali Samy\AppData\Local\pypoetry\Cache\virtualenvs\scout-radar-ER4m1cMX-py3.12\Lib\site-packages\mlflow\types\utils.py:440: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have mi

ZeroR Classifier
              precision    recall  f1-score   support

           0       0.80      1.00      0.89       796
           1       0.00      0.00      0.00       205

    accuracy                           0.80      1001
   macro avg       0.40      0.50      0.44      1001
weighted avg       0.63      0.80      0.70      1001



c:\Users\Ali Samy\AppData\Local\pypoetry\Cache\virtualenvs\scout-radar-ER4m1cMX-py3.12\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\Ali Samy\AppData\Local\pypoetry\Cache\virtualenvs\scout-radar-ER4m1cMX-py3.12\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\Ali Samy\AppData\Local\pypoetry\Cache\virtualenvs\scout-radar-ER4m1cMX-py3.12\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels wi

In [23]:
from imblearn.over_sampling import SMOTE
smote = SMOTE(k_neighbors=5, random_state=42)
X_train, y_train = smote.fit_resample(X_train, y_train)
print(y_train.value_counts(normalize=True)*100)

2026/05/03 14:51:07 INFO mlflow.utils.autologging_utils: Created MLflow autologging run with ID '95cf0db4ee3f4b9ba434346ade5c993a', which will track hyperparameters, performance metrics, model artifacts, and lineage information for the current sklearn workflow
2026/05/03 14:51:07 WARNING mlflow.sklearn: Failed to infer model signature: the trained model does not have a `predict` or `transform` function, which is required in order to infer the signature
2026/05/03 14:51:08 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
2026/05/03 14:51:08 WARNING mlflow.sklearn: Model was missing function: predict. Not logging python_function flavor!
2026/05/03 14:51:12 WARNI

transfer_success
0    50.0
1    50.0
Name: proportion, dtype: float64


In [ ]:
# now I will use decision tree classifier
hyperparameters = [5, 10, 15, 20]
for h in hyperparameters:
    with mlflow.start_run(run_name=f"Decision Tree min_samples_split={h}", experiment_id=experiment_id):
        params = {
            'criterion': 'entropy',
            'max_depth': 41,
            'min_samples_split': h,
        }
        mlflow.log_params(params)
        dt = DecisionTreeClassifier(random_state=42, **params)
        dt.fit(X_train, y_train)
        y_pred = dt.predict(X_val)
        print("Decision Tree Classifier for min_samples_split =", h)
        mlflow.log_metric("accuracy", accuracy_score(y_val, y_pred))
        mlflow.log_metric("precision", precision_score(y_val, y_pred))
        mlflow.log_metric("recall", recall_score(y_val, y_pred))
        mlflow.log_metric("f1_score", f1_score(y_val, y_pred))
        mlflow.sklearn.log_model(sk_model=dt, name=f"decision_tree_min_samples_split_{h}")
        results_df = results_df.append({
            "model_name": f"Decision Tree min_samples_split={h}",
            "accuracy": accuracy_score(y_val, y_pred),
            "precision": precision_score(y_val, y_pred),
            "recall": recall_score(y_val, y_pred),
            "f1_score": f1_score(y_val, y_pred)
        }, ignore_index=True)
        with open(f"../models/decision_tree_{h}_min_samples_split.pkl", "wb") as f:
            pickle.dump(dt, f)

2026/05/03 14:52:28 INFO mlflow.utils.autologging_utils: Created MLflow autologging run with ID 'e107bb0ec1d5445eafd9b8feaf43fda3', which will track hyperparameters, performance metrics, model artifacts, and lineage information for the current sklearn workflow
2026/05/03 14:52:28 WARNING mlflow.utils.autologging_utils: MLflow autologging encountered a warning: "c:\Users\Ali Samy\AppData\Local\pypoetry\Cache\virtualenvs\scout-radar-ER4m1cMX-py3.12\Lib\site-packages\mlflow\types\utils.py:440: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have mi

Decision Tree Classifier for min_samples_split = 5
              precision    recall  f1-score   support

           0       0.83      0.83      0.83       796
           1       0.33      0.33      0.33       205

    accuracy                           0.72      1001
   macro avg       0.58      0.58      0.58      1001
weighted avg       0.72      0.72      0.72      1001



2026/05/03 14:52:35 WARNING mlflow.utils.autologging_utils: MLflow autologging encountered a warning: "c:\Users\Ali Samy\AppData\Local\pypoetry\Cache\virtualenvs\scout-radar-ER4m1cMX-py3.12\Lib\site-packages\mlflow\types\utils.py:440: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details."
2026/05/03 14:52:35 WARNING mlflow.utils.autologging_utils: MLflow autologging encountered a war

Decision Tree Classifier for min_samples_split = 10
              precision    recall  f1-score   support

           0       0.83      0.85      0.84       796
           1       0.36      0.33      0.34       205

    accuracy                           0.74      1001
   macro avg       0.59      0.59      0.59      1001
weighted avg       0.73      0.74      0.74      1001



2026/05/03 14:52:42 WARNING mlflow.utils.autologging_utils: MLflow autologging encountered a warning: "c:\Users\Ali Samy\AppData\Local\pypoetry\Cache\virtualenvs\scout-radar-ER4m1cMX-py3.12\Lib\site-packages\mlflow\types\utils.py:440: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details."
2026/05/03 14:52:42 WARNING mlflow.utils.autologging_utils: MLflow autologging encountered a war

Decision Tree Classifier for min_samples_split = 15
              precision    recall  f1-score   support

           0       0.83      0.84      0.84       796
           1       0.34      0.32      0.33       205

    accuracy                           0.74      1001
   macro avg       0.58      0.58      0.58      1001
weighted avg       0.73      0.74      0.73      1001



2026/05/03 14:52:48 WARNING mlflow.utils.autologging_utils: MLflow autologging encountered a warning: "c:\Users\Ali Samy\AppData\Local\pypoetry\Cache\virtualenvs\scout-radar-ER4m1cMX-py3.12\Lib\site-packages\mlflow\types\utils.py:440: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details."
2026/05/03 14:52:48 WARNING mlflow.utils.autologging_utils: MLflow autologging encountered a war

Decision Tree Classifier for min_samples_split = 20
              precision    recall  f1-score   support

           0       0.82      0.85      0.84       796
           1       0.33      0.29      0.31       205

    accuracy                           0.74      1001
   macro avg       0.58      0.57      0.57      1001
weighted avg       0.72      0.74      0.73      1001



In [ ]:
# now trying logistic regression
hyperparameters = [10,100,1000]
for h in hyperparameters:
    with mlflow.start_run(run_name=f"Logistic Regression max_iter={h}", experiment_id=experiment_id):
        params = {
            'C': 1.0,
            'max_iter': h,
            'solver': 'lbfgs',
            'random_state': 42
        }
        mlflow.log_params(params)
        lr = LogisticRegression(**params)
        lr.fit(X_train,y_train)
        y_pred = lr.predict(X_val)
        print("LogisticRegression for max_iter =", h)
        mlflow.log_metric("accuracy", accuracy_score(y_val, y_pred))
        mlflow.log_metric("precision", precision_score(y_val, y_pred))
        mlflow.log_metric("recall", recall_score(y_val, y_pred))
        mlflow.log_metric("f1_score", f1_score(y_val, y_pred))
        mlflow.sklearn.log_model(sk_model=lr, name=f"logistic_regression_max_iter_{h}")
        results_df = results_df.append({
            "model_name": f"Logistic Regression max_iter={h}",
            "accuracy": accuracy_score(y_val, y_pred),
            "precision": precision_score(y_val, y_pred),
            "recall": recall_score(y_val, y_pred),
            "f1_score": f1_score(y_val, y_pred)
        }, ignore_index=True)
        with open(f"../models/logistic_regression_{h}_max_iter.pkl", "wb") as f:        
            pickle.dump(lr, f)

2026/05/03 14:55:14 INFO mlflow.utils.autologging_utils: Created MLflow autologging run with ID '99eba4b74dce4479bcbed29348b45cab', which will track hyperparameters, performance metrics, model artifacts, and lineage information for the current sklearn workflow
2026/05/03 14:55:14 WARNING mlflow.utils.autologging_utils: MLflow autologging encountered a warning: "c:\Users\Ali Samy\AppData\Local\pypoetry\Cache\virtualenvs\scout-radar-ER4m1cMX-py3.12\Lib\site-packages\mlflow\types\utils.py:440: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have mi

LogisticRegression for max_iter = 10
              precision    recall  f1-score   support

           0       0.86      0.72      0.78       796
           1       0.33      0.55      0.41       205

    accuracy                           0.68      1001
   macro avg       0.60      0.63      0.60      1001
weighted avg       0.75      0.68      0.71      1001



2026/05/03 14:55:21 WARNING mlflow.utils.autologging_utils: MLflow autologging encountered a warning: "c:\Users\Ali Samy\AppData\Local\pypoetry\Cache\virtualenvs\scout-radar-ER4m1cMX-py3.12\Lib\site-packages\mlflow\types\utils.py:440: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details."
c:\Users\Ali Samy\AppData\Local\pypoetry\Cache\virtualenvs\scout-radar-ER4m1cMX-py3.12\Lib\site-

LogisticRegression for max_iter = 100
              precision    recall  f1-score   support

           0       0.85      0.83      0.84       796
           1       0.39      0.42      0.40       205

    accuracy                           0.75      1001
   macro avg       0.62      0.62      0.62      1001
weighted avg       0.75      0.75      0.75      1001



2026/05/03 14:55:28 WARNING mlflow.utils.autologging_utils: MLflow autologging encountered a warning: "c:\Users\Ali Samy\AppData\Local\pypoetry\Cache\virtualenvs\scout-radar-ER4m1cMX-py3.12\Lib\site-packages\mlflow\types\utils.py:440: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details."
c:\Users\Ali Samy\AppData\Local\pypoetry\Cache\virtualenvs\scout-radar-ER4m1cMX-py3.12\Lib\site-

LogisticRegression for max_iter = 1000
              precision    recall  f1-score   support

           0       0.85      0.79      0.82       796
           1       0.36      0.45      0.40       205

    accuracy                           0.72      1001
   macro avg       0.60      0.62      0.61      1001
weighted avg       0.75      0.72      0.73      1001



In [ ]:
# now trying linear svm model
hyperparameters = [1,10,50,100]
for h in hyperparameters:
    with mlflow.start_run(run_name=f"Linear SVM C={h}", experiment_id=experiment_id):
        params = {
            'C': h,
            'random_state': 42
        }
        mlflow.log_params(params)
        model_svm = svm.LinearSVC(**params)
        model_svm.fit(X_train,y_train)
        y_pred = model_svm.predict(X_val)
        print("Linear SVM for C =", h)
        mlflow.log_metric("accuracy", accuracy_score(y_val, y_pred))
        mlflow.log_metric("precision", precision_score(y_val, y_pred))
        mlflow.log_metric("recall", recall_score(y_val, y_pred))
        mlflow.log_metric("f1_score", f1_score(y_val, y_pred))
        mlflow.sklearn.log_model(sk_model=model_svm, name=f"linear_svm_C_{h}")
        results_df = results_df.append({
            "model_name": f"Linear SVM C={h}",
            "accuracy": accuracy_score(y_val, y_pred),
            "precision": precision_score(y_val, y_pred),
            "recall": recall_score(y_val, y_pred),
            "f1_score": f1_score(y_val, y_pred)
        }, ignore_index=True)
        with open(f"../models/linear_svm_{h}_C.pkl", "wb") as f:
            pickle.dump(model_svm, f)

2026/05/03 14:56:12 INFO mlflow.utils.autologging_utils: Created MLflow autologging run with ID '375ee67669d948beb4cb10b0c150c603', which will track hyperparameters, performance metrics, model artifacts, and lineage information for the current sklearn workflow
2026/05/03 14:56:13 WARNING mlflow.utils.autologging_utils: MLflow autologging encountered a warning: "c:\Users\Ali Samy\AppData\Local\pypoetry\Cache\virtualenvs\scout-radar-ER4m1cMX-py3.12\Lib\site-packages\mlflow\types\utils.py:440: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have mi

Linear SVM for C = 1
              precision    recall  f1-score   support

           0       0.84      0.82      0.83       796
           1       0.37      0.40      0.38       205

    accuracy                           0.74      1001
   macro avg       0.60      0.61      0.61      1001
weighted avg       0.74      0.74      0.74      1001



2026/05/03 14:56:19 WARNING mlflow.utils.autologging_utils: MLflow autologging encountered a warning: "c:\Users\Ali Samy\AppData\Local\pypoetry\Cache\virtualenvs\scout-radar-ER4m1cMX-py3.12\Lib\site-packages\mlflow\types\utils.py:440: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details."
2026/05/03 14:56:19 WARNING mlflow.utils.autologging_utils: MLflow autologging encountered a war

Linear SVM for C = 10
              precision    recall  f1-score   support

           0       0.85      0.84      0.84       796
           1       0.40      0.42      0.41       205

    accuracy                           0.75      1001
   macro avg       0.63      0.63      0.63      1001
weighted avg       0.76      0.75      0.76      1001



2026/05/03 14:56:26 WARNING mlflow.utils.autologging_utils: MLflow autologging encountered a warning: "c:\Users\Ali Samy\AppData\Local\pypoetry\Cache\virtualenvs\scout-radar-ER4m1cMX-py3.12\Lib\site-packages\mlflow\types\utils.py:440: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details."
2026/05/03 14:56:26 WARNING mlflow.utils.autologging_utils: MLflow autologging encountered a war

Linear SVM for C = 50
              precision    recall  f1-score   support

           0       0.85      0.85      0.85       796
           1       0.40      0.40      0.40       205

    accuracy                           0.76      1001
   macro avg       0.62      0.62      0.62      1001
weighted avg       0.76      0.76      0.76      1001



2026/05/03 14:56:33 WARNING mlflow.utils.autologging_utils: MLflow autologging encountered a warning: "c:\Users\Ali Samy\AppData\Local\pypoetry\Cache\virtualenvs\scout-radar-ER4m1cMX-py3.12\Lib\site-packages\mlflow\types\utils.py:440: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details."
2026/05/03 14:56:33 WARNING mlflow.utils.autologging_utils: MLflow autologging encountered a war

Linear SVM for C = 100
              precision    recall  f1-score   support

           0       0.85      0.80      0.82       796
           1       0.36      0.45      0.40       205

    accuracy                           0.73      1001
   macro avg       0.61      0.62      0.61      1001
weighted avg       0.75      0.73      0.74      1001



In [ ]:
# now tring random forest classifier
hyperparameters = [5,10,15,20]
for h in hyperparameters:
    with mlflow.start_run(run_name=f"Random Forest n_estimators={h}", experiment_id=experiment_id):
        params = {
            'n_estimators': h,
            'random_state': 42
        }
        mlflow.log_params(params)
        model_rf = RandomForestClassifier(**params)
        model_rf.fit(X_train,y_train)
        y_pred = model_rf.predict(X_val)
        print("Random Forest Classifier for n_estimators =", h)
        mlflow.log_metric("accuracy", accuracy_score(y_val, y_pred))
        mlflow.log_metric("precision", precision_score(y_val, y_pred))
        mlflow.log_metric("recall", recall_score(y_val, y_pred))
        mlflow.log_metric("f1_score", f1_score(y_val, y_pred))
        mlflow.sklearn.log_model(sk_model=model_rf, name=f"random_forest_n_estimators_{h}")
        results_df = results_df.append({
            "model_name": f"Random Forest n_estimators={h}",
            "accuracy": accuracy_score(y_val, y_pred),
            "precision": precision_score(y_val, y_pred),
            "recall": recall_score(y_val, y_pred),
            "f1_score": f1_score(y_val, y_pred)
        }, ignore_index=True)
        with open(f"../models/random_forest_{h}_estimators.pkl", "wb") as f:
            pickle.dump(model_rf, f)

2026/05/03 14:57:06 INFO mlflow.utils.autologging_utils: Created MLflow autologging run with ID '22cc0c3a5c084ed6b57bfbec83dcf930', which will track hyperparameters, performance metrics, model artifacts, and lineage information for the current sklearn workflow
2026/05/03 14:57:06 WARNING mlflow.utils.autologging_utils: MLflow autologging encountered a warning: "c:\Users\Ali Samy\AppData\Local\pypoetry\Cache\virtualenvs\scout-radar-ER4m1cMX-py3.12\Lib\site-packages\mlflow\types\utils.py:440: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have mi

Random Forest Classifier for n_estimators = 5
              precision    recall  f1-score   support

           0       0.82      0.91      0.86       796
           1       0.38      0.21      0.27       205

    accuracy                           0.77      1001
   macro avg       0.60      0.56      0.57      1001
weighted avg       0.73      0.77      0.74      1001



2026/05/03 14:57:13 WARNING mlflow.utils.autologging_utils: MLflow autologging encountered a warning: "c:\Users\Ali Samy\AppData\Local\pypoetry\Cache\virtualenvs\scout-radar-ER4m1cMX-py3.12\Lib\site-packages\mlflow\types\utils.py:440: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details."
2026/05/03 14:57:13 WARNING mlflow.utils.autologging_utils: MLflow autologging encountered a war

Random Forest Classifier for n_estimators = 10
              precision    recall  f1-score   support

           0       0.81      0.96      0.88       796
           1       0.48      0.15      0.23       205

    accuracy                           0.79      1001
   macro avg       0.65      0.55      0.56      1001
weighted avg       0.75      0.79      0.75      1001



2026/05/03 14:57:25 WARNING mlflow.utils.autologging_utils: MLflow autologging encountered a warning: "c:\Users\Ali Samy\AppData\Local\pypoetry\Cache\virtualenvs\scout-radar-ER4m1cMX-py3.12\Lib\site-packages\mlflow\types\utils.py:440: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details."
2026/05/03 14:57:25 WARNING mlflow.utils.autologging_utils: MLflow autologging encountered a war

Random Forest Classifier for n_estimators = 15
              precision    recall  f1-score   support

           0       0.82      0.94      0.88       796
           1       0.48      0.20      0.29       205

    accuracy                           0.79      1001
   macro avg       0.65      0.57      0.58      1001
weighted avg       0.75      0.79      0.76      1001



2026/05/03 14:57:32 WARNING mlflow.utils.autologging_utils: MLflow autologging encountered a warning: "c:\Users\Ali Samy\AppData\Local\pypoetry\Cache\virtualenvs\scout-radar-ER4m1cMX-py3.12\Lib\site-packages\mlflow\types\utils.py:440: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details."
2026/05/03 14:57:32 WARNING mlflow.utils.autologging_utils: MLflow autologging encountered a war

Random Forest Classifier for n_estimators = 20
              precision    recall  f1-score   support

           0       0.82      0.97      0.89       796
           1       0.59      0.18      0.28       205

    accuracy                           0.81      1001
   macro avg       0.70      0.57      0.58      1001
weighted avg       0.77      0.81      0.76      1001



In [ ]:
# now trying XGBoost classifier
hyperparameters = [5,10,20,50]
for h in hyperparameters:
    with mlflow.start_run(run_name=f"XGBoost n_estimators={h}", experiment_id=experiment_id):
        params = {
            'n_estimators': h,
            'max_depth': 41,
            'learning_rate': 0.1,
        }
        mlflow.log_params(params)
        xgb = XGBClassifier(**params)
        xgb.fit(X_train, y_train)
        y_pred = xgb.predict(X_val)
        print("XGBoost Classifier for n_estimators =", h)
        mlflow.log_metric("accuracy", accuracy_score(y_val, y_pred))
        mlflow.log_metric("precision", precision_score(y_val, y_pred))
        mlflow.log_metric("recall", recall_score(y_val, y_pred))
        mlflow.log_metric("f1_score", f1_score(y_val, y_pred))
        mlflow.xgboost.log_model(sk_model=xgb, name=f"xgboost_n_estimators_{h}")
        results_df = results_df.append({
            "model_name": f"XGBoost n_estimators={h}",
            "accuracy": accuracy_score(y_val, y_pred),
            "precision": precision_score(y_val, y_pred),
            "recall": recall_score(y_val, y_pred),
            "f1_score": f1_score(y_val, y_pred)
        }, ignore_index=True)
        with open(f"../models/xgb_model_{h}_estimators.pkl", "wb") as f:
            pickle.dump(xgb, f)

XGBoost Classifier for n_estimators = 5
              precision    recall  f1-score   support

           0       0.83      0.92      0.87       796
           1       0.47      0.29      0.36       205

    accuracy                           0.79      1001
   macro avg       0.65      0.60      0.62      1001
weighted avg       0.76      0.79      0.77      1001

XGBoost Classifier for n_estimators = 10
              precision    recall  f1-score   support

           0       0.83      0.93      0.88       796
           1       0.49      0.26      0.34       205

    accuracy                           0.79      1001
   macro avg       0.66      0.59      0.61      1001
weighted avg       0.76      0.79      0.77      1001

XGBoost Classifier for n_estimators = 20
              precision    recall  f1-score   support

           0       0.82      0.95      0.88       796
           1       0.51      0.21      0.30       205

    accuracy                           0.80      1001
   mac

In [ ]:
print(results_df.sort_values(by="f1_score", ascending=False))

In [33]:
y_test = data_dict['test']['transfer_success']
X_test = data_dict['test'].drop(columns=['transfer_success'])

In [ ]:
# Load model
run_id = 'a74aa6166c014c009699b9c81c05918e'
model_name = 'random_forest_classifier'
model_uri = f'runs:/{run_id}/{model_name}'
dt = mlflow.sklearn.load_model(model_uri=model_uri)
dt.fit(X_train, y_train)
y_pred = dt.predict(X_test)